In [7]:
import os
import glob
import numpy as np
from scipy.spatial import cKDTree
from scipy.linalg import eigh
import networkx as nx
from collections import defaultdict

In [11]:
def triangulate_polygon(indices_1based):
    """
    Triangulate a convex polygon using fan triangulation.
    indices_1based: list of vertex indices (starting from 1)
    Returns list of triangles in 0-based indexing.
    """
    n = len(indices_1based)
    if n < 3:
        return []
    triangles = []
    v0 = indices_1based[0] - 1
    for i in range(1, n-1):
        v1 = indices_1based[i] - 1
        v2 = indices_1based[i+1] - 1
        triangles.append([v0, v1, v2])
    return triangles

def read_obj(filepath):
    """
    Read OBJ file. Extracts vertices and faces.
    Supports polygons of any size (triangulated on the fly).
    Returns vertices (Nx3) and faces (list of triangles).
    """
    vertices = []
    polygons = []

    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.split()
                v = [float(parts[1]), float(parts[2]), float(parts[3])]
                vertices.append(v)
            elif line.startswith('f '):
                parts = line.split()[1:]
                # Extract vertex indices (ignore texture and normal indices)
                face = [int(p.split('/')[0]) for p in parts]
                polygons.append(face)

    vertices = np.array(vertices, dtype=np.float64)

    # Triangulate all polygons
    faces_tri = []
    for poly in polygons:
        if len(poly) == 3:
            faces_tri.append([poly[0]-1, poly[1]-1, poly[2]-1])
        elif len(poly) == 4:
            faces_tri.append([poly[0]-1, poly[1]-1, poly[2]-1])
            faces_tri.append([poly[0]-1, poly[2]-1, poly[3]-1])
        else:
            tris = triangulate_polygon(poly)
            faces_tri.extend(tris)
    return vertices, np.array(faces_tri)

def compute_vertex_normals(vertices, faces):
    """
    Compute vertex normals as area-weighted average of adjacent triangle normals.
    """
    normals = np.zeros_like(vertices)
    tri_normals = np.zeros((len(faces), 3))
    tri_areas = np.zeros(len(faces))

    for i, face in enumerate(faces):
        v0, v1, v2 = vertices[face]
        e1 = v1 - v0
        e2 = v2 - v0
        n = np.cross(e1, e2)
        area = np.linalg.norm(n) / 2.0
        if area > 1e-12:
            n = n / (2.0 * area)
        else:
            n = np.zeros(3)
        tri_normals[i] = n
        tri_areas[i] = area

    for i, face in enumerate(faces):
        for v_idx in face:
            normals[v_idx] += tri_normals[i] * tri_areas[i]

    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normals = normals / norms
    return normals

def density_analysis(points):
    """
    Evaluate point cloud density.
    Returns mean nearest neighbor distance, coefficient of variation,
    uniformity qualifier, and density qualifier.
    """
    tree = cKDTree(points)
    dists, _ = tree.query(points, k=2)
    nearest_dists = dists[:, 1]
    mean_dist = np.mean(nearest_dists)
    std_dist = np.std(nearest_dists)
    cv = std_dist / mean_dist if mean_dist > 0 else 0

    if cv < 0.3:
        uniformity = "high"
    elif cv < 0.6:
        uniformity = "medium"
    else:
        uniformity = "low"

    bbox = np.max(points, axis=0) - np.min(points, axis=0)
    scale = np.linalg.norm(bbox)
    rel_density = mean_dist / scale if scale > 0 else 0
    if rel_density < 0.01:
        density_qual = "high"
    elif rel_density < 0.05:
        density_qual = "medium"
    else:
        density_qual = "low"

    return mean_dist, cv, uniformity, density_qual

def shape_analysis(points):
    """
    Perform PCA to determine shape of point distribution.
    Returns normalized eigenvalues and shape interpretation.
    """
    centroid = np.mean(points, axis=0)
    centered = points - centroid
    cov = np.cov(centered.T)
    eigvals, _ = eigh(cov)
    eigvals = eigvals[::-1]
    eigvals_norm = eigvals / np.sum(eigvals)

    if eigvals_norm[0] > 0.8 and eigvals_norm[1] < 0.1:
        shape = "linear structure"
    elif eigvals_norm[0] > 0.6 and eigvals_norm[1] > 0.2:
        shape = "planar structure"
    else:
        shape = "volumetric shape"
    return eigvals_norm, shape

def curvature_analysis(points, k_neighbors=30):
    """
    Estimate surface curvature using local PCA.
    Returns mean curvature and surface type classification.
    """
    tree = cKDTree(points)
    curvatures = []
    for i, p in enumerate(points):
        indices = tree.query(p, k=k_neighbors)[1]
        neighbors = points[indices]
        centroid = np.mean(neighbors, axis=0)
        centered = neighbors - centroid
        cov = np.cov(centered.T)
        eigvals, _ = eigh(cov)
        eigvals = eigvals[::-1]
        curvature = eigvals[2] / (eigvals[0] + eigvals[1] + eigvals[2] + 1e-12)
        curvatures.append(curvature)
    mean_curv = np.mean(curvatures)
    if mean_curv < 0.02:
        surface_type = "smooth surface"
    elif mean_curv < 0.08:
        surface_type = "slightly curved"
    elif mean_curv < 0.2:
        surface_type = "highly curved"
    else:
        surface_type = "complex (combined)"
    return mean_curv, surface_type

def normal_consistency(vertices, faces, vertex_normals):
    """
    Evaluate consistency of normals across edges.
    Returns mean angle between adjacent normals,
    proportion of sharp edges (>30°), and consistency interpretation.
    """
    edges = set()
    for face in faces:
        for i in range(3):
            v1, v2 = face[i], face[(i+1)%3]
            edge = tuple(sorted((v1, v2)))
            edges.add(edge)

    angles = []
    for v1, v2 in edges:
        n1 = vertex_normals[v1]
        n2 = vertex_normals[v2]
        cos_angle = np.clip(np.dot(n1, n2), -1.0, 1.0)
        angle = np.arccos(cos_angle) * 180.0 / np.pi
        angles.append(angle)
    angles = np.array(angles)
    mean_angle = np.mean(angles)
    sharp_ratio = np.sum(angles > 30) / len(angles) if len(angles) > 0 else 0

    if mean_angle < 10:
        consistency = "high consistency (planar/smooth surfaces)"
    elif mean_angle < 25:
        consistency = "medium consistency (cylindrical/spherical)"
    else:
        consistency = "low consistency (complex geometry or noise)"

    return mean_angle, sharp_ratio, consistency

def topology_analysis(vertices, faces):
    """
    Analyze topological connectivity of the mesh.
    Returns number of connected components, presence of isolated clusters,
    average vertex degree, and presence of holes (boundary edges).
    """
    G = nx.Graph()
    G.add_nodes_from(range(len(vertices)))
    for face in faces:
        for i in range(3):
            v1, v2 = face[i], face[(i+1)%3]
            G.add_edge(v1, v2)

    components = list(nx.connected_components(G))
    num_components = len(components)
    isolated = any(len(comp) == 1 for comp in components)

    degrees = [deg for _, deg in G.degree()]
    avg_degree = np.mean(degrees) if degrees else 0

    edge_count = defaultdict(int)
    for face in faces:
        for i in range(3):
            v1, v2 = face[i], face[(i+1)%3]
            edge = tuple(sorted((v1, v2)))
            edge_count[edge] += 1
    boundary_edges = [e for e, cnt in edge_count.items() if cnt == 1]
    has_holes = len(boundary_edges) > 0

    return num_components, isolated, avg_degree, has_holes

In [14]:
def analyze_segment(vertices, faces, filename):
    """
    Perform all geometric analyses on a single segment (mesh).
    Prints results.
    """
    if len(vertices) == 0:
        print("No vertices to analyze.")
        return

    print(f"Vertices: {len(vertices)}, Triangles: {len(faces)}")

    # 6.1 Point density
    mean_dist, cv, uniformity, density_qual = density_analysis(vertices)
    print("\n=== 6.1 Point density ===")
    print(f"Mean distance to nearest neighbor: {mean_dist:.4f}")
    print(f"Coefficient of variation: {cv:.3f} -> uniformity: {uniformity}")
    print(f"Density qualifier: {density_qual}")

    # 6.2 Distribution shape
    eigvals_norm, shape = shape_analysis(vertices)
    print("\n=== 6.2 Distribution shape ===")
    print(f"Normalized eigenvalues: {eigvals_norm[0]:.3f}, {eigvals_norm[1]:.3f}, {eigvals_norm[2]:.3f}")
    print(f"Interpretation: {shape}")

    # 6.3 Surface structure (curvature)
    mean_curv, surface_type = curvature_analysis(vertices)
    print("\n=== 6.3 Surface structure ===")
    print(f"Mean local curvature: {mean_curv:.4f}")
    print(f"Classification: {surface_type}")

    # 6.4 Normal consistency
    vertex_normals = compute_vertex_normals(vertices, faces)
    mean_angle, sharp_ratio, consistency = normal_consistency(vertices, faces, vertex_normals)
    print("\n=== 6.4 Normal consistency ===")
    print(f"Mean angle between adjacent vertex normals: {mean_angle:.2f}°")
    print(f"Fraction of edges with angle >30°: {sharp_ratio*100:.1f}%")
    print(f"Interpretation: {consistency}")

    # 6.5 Topological connectivity
    num_comp, isolated, avg_deg, has_holes = topology_analysis(vertices, faces)
    print("\n=== 6.5 Topological connectivity ===")
    print(f"Number of connected components: {num_comp}")
    print(f"Isolated clusters present: {'yes' if isolated else 'no'}")
    print(f"Average vertex degree: {avg_deg:.2f}")
    print(f"Holes (boundary edges) present: {'yes' if has_holes else 'no'}")
    if num_comp == 1 and not isolated and not has_holes:
        integrity = "fully connected, no holes"
    else:
        integrity = "segment integrity is compromised"
    print(f"Segment integrity: {integrity}")

    # 6.6 Summary
    print("\n=== 6.6 Summary ===")
    print(f"  Point density: {density_qual}, uniformity: {uniformity}")
    print(f"  Distribution shape: {shape}")
    print(f"  Surface type: {surface_type}")
    print(f"  Normal consistency: {consistency}")
    print(f"  Topology: {num_comp} component(s), {'holes' if has_holes else 'no holes'}, integrity: {integrity}")
    print("\nThese results are used for segment classification and surface reconstruction method selection.")

def process_files(folder_path):
    """
    Process all .obj files in the given folder (including subfolders).
    """
    obj_files = glob.glob(os.path.join(folder_path, '**', '*.obj'), recursive=True)
    if not obj_files:
        print(f"No .obj files found in {folder_path}")
        return

    for obj_file in obj_files:
        print(f"\n{'='*70}")
        print(f"Analyzing file: {obj_file}")
        vertices, faces = read_obj(obj_file)
        analyze_segment(vertices, faces, obj_file)

In [15]:
process_files("dataset_task3")


Analyzing file: dataset_task3\13.01.obj
Vertices: 1204, Triangles: 2408

=== 6.1 Point density ===
Mean distance to nearest neighbor: 1.8498
Coefficient of variation: 0.666 -> uniformity: low
Density qualifier: medium

=== 6.2 Distribution shape ===
Normalized eigenvalues: 0.416, 0.388, 0.196
Interpretation: volumetric shape

=== 6.3 Surface structure ===
Mean local curvature: 0.0603
Classification: slightly curved

=== 6.4 Normal consistency ===
Mean angle between adjacent vertex normals: 23.05°
Fraction of edges with angle >30°: 17.7%
Interpretation: medium consistency (cylindrical/spherical)

=== 6.5 Topological connectivity ===
Number of connected components: 1
Isolated clusters present: no
Average vertex degree: 6.00
Holes (boundary edges) present: no
Segment integrity: fully connected, no holes

=== 6.6 Summary ===
  Point density: medium, uniformity: low
  Distribution shape: volumetric shape
  Surface type: slightly curved
  Normal consistency: medium consistency (cylindrical/